# Chapter 15: Oracles — Judging Agent Behavior Geometrically

**Learning objectives:**
- Understand the seven oracle types and which GMS primitive each uses
- Compose oracles via `CompositeOracle` for multi-signal verdicts
- Use `TraceOracle` (holonomy) and `SideEffectOracle` (tension) for tool-sequence and state-change verification
- Create oracles from configuration via `make_oracle()` factory

---
## Preamble

Chapters 11–14 built the black-box testing infrastructure: adapters capture traces, the gateway injects faults, the policy engine enforces rules and the environment simulator provides deterministic world-state. But none of these layers answer the central question: **did the agent behave correctly?**

That is the job of the **oracle**. An oracle takes a scenario (what we expect), a trace (what the agent did) and optional environment state, then returns a structured verdict: pass or fail, with confidence and evidence.

GMSH provides seven oracle types, each leveraging a different GMS geometric primitive:

| Oracle | GMS Primitive | Judges |
|---|---|---|
| `ExactOutcomeOracle` | `score_triple()` | Single correct answer + geodesic |
| `AcceptableSetOracle` | geodesic to nearest | Best-of-set matching |
| `ConstraintOracle` | predicates | Must-have / must-not-have |
| `TraceOracle` | `check_holonomy()` | Tool-call sequence patterns |
| `SideEffectOracle` | `tension_energy()` | Environment state changes |
| `HumanReviewedOracle` | rubric-driven | LLM/human review |
| `CompositeOracle` | weighted combination | Multi-signal aggregation |

In [ ]:
# === Preamble (all notebooks) ===
import warnings
warnings.filterwarnings("ignore")

import torch
import numpy as np

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [ ]:
from importlib.resources import files
# Load the Chapter 4 store
from knowlytix.harness.testing import DOEGMSBenchmark, DOEHarnessConfig

config = DOEHarnessConfig(
    markdown_path=str(files("knowlytix.harness.testing").joinpath("tests/model_risk_assessment.md")),
    stores_dir="../lab_stores",
    store_name="ch04_ground_truth",
    n_runs=4,
    ingest_epochs=200,
    device=str(DEVICE),
)
bench = DOEGMSBenchmark(config)
store = bench.ingest()
print(f"Store: {store.stats()['entities']} entities, "
      f"{store.stats()['triples']} triples, "
      f"{store.stats()['enm_entries']} ENM")

---

## OracleVerdict: The Structured Result

Every oracle returns an `OracleVerdict` with a uniform schema:

| Field | Type | Purpose |
|---|---|---|
| `oracle_type` | str | Which oracle produced this verdict |
| `passed` | bool | Whether the SUT behavior is acceptable |
| `confidence` | float | [0, 1] confidence level |
| `detail` | str | Human-readable explanation |
| `evidence` | list[str] | Supporting evidence items |
| `raw_scores` | dict[str, float] | Named scores from oracle internals |

---

## The Seven Oracle Types

Each oracle implements `OracleBase.evaluate(scenario, trace, env_state_before, env_state_after) -> OracleVerdict`. The `Scenario` dataclass carries the expected behavior (ground truth, constraints, trace pattern, expected side effects) and the `SUTTrace` carries the actual behavior (raw output, tool calls, latency).

In [ ]:
# Create a Scenario and SUTTrace for oracle demos
from knowlytix.harness.testing.scenarios import Scenario
from knowlytix.harness.testing.adapters import SUTTrace, ToolCallRecord
from knowlytix.harness.testing.oracles import (
    ExactOutcomeOracle, AcceptableSetOracle,
    ConstraintOracle, TraceOracle, SideEffectOracle,
    CompositeOracle, make_oracle,
)

# Scenario: "What is the risk rating of MDL-001?"
scenario = Scenario(
    scenario_id="S-001",
    name="MDL-001 risk rating",
    question_text="What is the risk rating of MDL-001?",
    ground_truth="High",
    metadata={"acceptable_set": ["High", "high", "HIGH"]},
    trace_pattern="data_lookup > risk_check, !dangerous_delete",
    expected_tool_sequence=["data_lookup", "risk_check"],
    constraints=[
        {"type": "must_contain", "value": "High"},
        {"type": "must_not_contain", "value": "I don't know"},
        {"type": "max_tool_calls", "value": 5},
        {"type": "required_tools", "value": ["data_lookup"]},
        {"type": "forbidden_tools", "value": ["dangerous_delete"]},
    ],
    expected_side_effects={"tickets_created": "__unchanged__"},
)

# Good trace: correct answer, correct tool sequence
good_trace = SUTTrace(
    raw_output="The risk rating of MDL-001 is High.",
    tool_calls=[
        ToolCallRecord(tool_name="data_lookup", arguments={"entity": "MDL-001"},
                       result="CreditScore-XGB, risk=High", latency_ms=0.5),
        ToolCallRecord(tool_name="risk_check", arguments={"model": "MDL-001"},
                       result="High", latency_ms=0.3),
    ],
    latency_ms=1.2,
)

# Bad trace: wrong answer, missing risk_check, uses forbidden tool
bad_trace = SUTTrace(
    raw_output="I don't know the risk rating.",
    tool_calls=[
        ToolCallRecord(tool_name="dangerous_delete", arguments={"target": "all"},
                       result="Deleted", latency_ms=0.1),
    ],
    latency_ms=0.5,
)

print(f"Scenario: {scenario.name}")
print(f"Good trace: {good_trace.raw_output!r}")
print(f"Bad trace:  {bad_trace.raw_output!r}")

In [4]:
# 1. ExactOutcomeOracle — single correct answer
exact = ExactOutcomeOracle(store=store)

v_good = exact.evaluate(scenario, good_trace)
v_bad = exact.evaluate(scenario, bad_trace)

print("=== ExactOutcomeOracle ===")
print(f"Good trace: passed={v_good.passed}, confidence={v_good.confidence:.3f}")
print(f"  detail: {v_good.detail}")
print(f"  scores: {v_good.raw_scores}")
print(f"\nBad trace:  passed={v_bad.passed}, confidence={v_bad.confidence:.3f}")
print(f"  detail: {v_bad.detail}")
print(f"  scores: {v_bad.raw_scores}")

=== ExactOutcomeOracle ===
Good trace: passed=True, confidence=1.000
  detail: exact_match=True, gms_score=None
  scores: {'exact_match': 1.0}

Bad trace:  passed=False, confidence=0.000
  detail: exact_match=False, gms_score=None
  scores: {'exact_match': 0.0}


In [5]:
# 2. AcceptableSetOracle — set of valid outputs
acceptable = AcceptableSetOracle(store=store)

v_good_a = acceptable.evaluate(scenario, good_trace)
v_bad_a = acceptable.evaluate(scenario, bad_trace)

print("=== AcceptableSetOracle ===")
print(f"Good trace: passed={v_good_a.passed}, confidence={v_good_a.confidence:.3f}")
print(f"  detail: {v_good_a.detail}")
print(f"\nBad trace:  passed={v_bad_a.passed}, confidence={v_bad_a.confidence:.3f}")
print(f"  detail: {v_bad_a.detail}")

=== AcceptableSetOracle ===
Good trace: passed=True, confidence=1.000
  detail: best_match=High, score=1.000, checked 3 candidates

Bad trace:  passed=False, confidence=0.000
  detail: best_match=None, score=0.000, checked 3 candidates


In [6]:
# 3. ConstraintOracle — behavioral constraints
constraint = ConstraintOracle(store=store)

v_good_c = constraint.evaluate(scenario, good_trace)
v_bad_c = constraint.evaluate(scenario, bad_trace)

print("=== ConstraintOracle ===")
print(f"Good trace: passed={v_good_c.passed}, confidence={v_good_c.confidence:.3f}")
print(f"  detail: {v_good_c.detail}")
print(f"  evidence: {v_good_c.evidence}")
print(f"\nBad trace:  passed={v_bad_c.passed}, confidence={v_bad_c.confidence:.3f}")
print(f"  detail: {v_bad_c.detail}")
print("  evidence:")
for e in v_bad_c.evidence:
    print(f"    - {e}")

=== ConstraintOracle ===
Good trace: passed=True, confidence=1.000
  detail: 5/5 constraints satisfied
  evidence: ['All constraints satisfied']

Bad trace:  passed=False, confidence=0.200
  detail: 1/5 constraints satisfied
  evidence:
    - Output missing required: 'High'
    - Output contains prohibited: 'I don't know'
    - Required tools missing: {'data_lookup'}
    - Forbidden tools used: {'dangerous_delete'}


In [7]:
# 4. TraceOracle — tool-call sequence verification
trace_oracle = TraceOracle(store=store)

v_good_t = trace_oracle.evaluate(scenario, good_trace)
v_bad_t = trace_oracle.evaluate(scenario, bad_trace)

print("=== TraceOracle ===")
print(f"Pattern: {scenario.trace_pattern!r}")
print(f"\nGood trace: passed={v_good_t.passed}, confidence={v_good_t.confidence:.3f}")
print(f"  detail: {v_good_t.detail}")
print(f"  scores: {v_good_t.raw_scores}")
print(f"\nBad trace:  passed={v_bad_t.passed}, confidence={v_bad_t.confidence:.3f}")
print(f"  detail: {v_bad_t.detail}")
print("  evidence:")
for e in v_bad_t.evidence:
    print(f"    - {e}")

=== TraceOracle ===
Pattern: 'data_lookup > risk_check, !dangerous_delete'

Good trace: passed=True, confidence=1.000
  detail: trace: 2 calls, 0 violations
  scores: {'checks': 2.0, 'violations': 0.0}

Bad trace:  passed=False, confidence=0.000
  detail: trace: 1 calls, 2 violations
  evidence:
    - Required tool 'data_lookup' not in trace
    - Prohibited tool found: 'dangerous_delete'


In [8]:
# 5. SideEffectOracle — environment state change verification
side_effect = SideEffectOracle(store=store)

# State unchanged — should pass
state_before = {"tickets_created": 0, "query_count": 0}
state_after_good = {"tickets_created": 0, "query_count": 1}

v_side_good = side_effect.evaluate(scenario, good_trace,
                                    state_before, state_after_good)
print("=== SideEffectOracle ===")
print(f"State unchanged: passed={v_side_good.passed}, "
      f"confidence={v_side_good.confidence:.3f}")
print(f"  detail: {v_side_good.detail}")

# State changed — should fail
state_after_bad = {"tickets_created": 5, "query_count": 1}
v_side_bad = side_effect.evaluate(scenario, bad_trace,
                                   state_before, state_after_bad)
print(f"\nState changed:   passed={v_side_bad.passed}, "
      f"confidence={v_side_bad.confidence:.3f}")
print(f"  detail: {v_side_bad.detail}")
print(f"  evidence: {v_side_bad.evidence}")

=== SideEffectOracle ===
State unchanged: passed=True, confidence=1.000
  detail: 1/1 side effects verified

State changed:   passed=False, confidence=0.000
  detail: 0/1 side effects verified
  evidence: ["State 'tickets_created' changed unexpectedly: {'before': 0, 'after': 5}"]


---

## Lab 15.1 — Compose Oracles for Agent Verification

The `CompositeOracle` combines multiple oracle types into a single weighted verdict. Each sub-oracle runs independently; the composite computes a weighted confidence average and optionally fails if any sub-oracle fails (`fail_on_any_critical=True`).

In [9]:
# Lab 15.1a — Build a CompositeOracle with three sub-oracles
composite = CompositeOracle(
    oracles=[
        (ExactOutcomeOracle(store=store), 0.5),    # 50% on correctness
        (ConstraintOracle(store=store), 0.2),       # 20% on constraints
        (TraceOracle(store=store), 0.3),             # 30% on process
    ],
    threshold=0.5,
    fail_on_any_critical=True,
)

# Evaluate good trace
v_comp_good = composite.evaluate(scenario, good_trace)
print("=== CompositeOracle (good trace) ===")
print(f"  passed     : {v_comp_good.passed}")
print(f"  confidence : {v_comp_good.confidence:.3f}")
print(f"  detail     : {v_comp_good.detail}")
print("  scores:")
for k, v in sorted(v_comp_good.raw_scores.items()):
    print(f"    {k}: {v:.4f}")

=== CompositeOracle (good trace) ===
  passed     : True
  confidence : 1.000
  detail     : composite(3 oracles): avg_conf=1.000, threshold=0.5 | exact_outcome: PASS (conf=1.000); constraint: PASS (conf=1.000); trace: PASS (conf=1.000)
  scores:
    constraint.checks_passed: 5.0000
    constraint.total_checks: 5.0000
    constraint.violations: 0.0000
    exact_outcome.exact_match: 1.0000
    trace.checks: 2.0000
    trace.violations: 0.0000


In [10]:
# Lab 15.1b — Evaluate bad trace through the same composite
v_comp_bad = composite.evaluate(scenario, bad_trace)
print("=== CompositeOracle (bad trace) ===")
print(f"  passed     : {v_comp_bad.passed}")
print(f"  confidence : {v_comp_bad.confidence:.3f}")
print(f"  detail     : {v_comp_bad.detail}")
print(f"  evidence ({len(v_comp_bad.evidence)} items):")
for e in v_comp_bad.evidence[:8]:
    print(f"    - {e}")

=== CompositeOracle (bad trace) ===
  passed     : False
  confidence : 0.040
  detail     : composite(3 oracles): avg_conf=0.040, threshold=0.5 | exact_outcome: FAIL (conf=0.000); constraint: FAIL (conf=0.200); trace: FAIL (conf=0.000)
  evidence (8 items):
    - expected=High
    - actual=I don't know the risk rating.
    - Output missing required: 'High'
    - Output contains prohibited: 'I don't know'
    - Required tools missing: {'data_lookup'}
    - Forbidden tools used: {'dangerous_delete'}
    - Required tool 'data_lookup' not in trace
    - Prohibited tool found: 'dangerous_delete'


In [11]:
# Lab 15.1c — Compare verdicts side by side
print(f"{'Metric':<20s} {'Good Trace':>12s} {'Bad Trace':>12s}")
print("-" * 46)
print(f"{'passed':<20s} {str(v_comp_good.passed):>12s} {str(v_comp_bad.passed):>12s}")
print(f"{'confidence':<20s} {v_comp_good.confidence:>12.3f} {v_comp_bad.confidence:>12.3f}")

# Compare per-oracle scores
all_keys = sorted(set(v_comp_good.raw_scores) | set(v_comp_bad.raw_scores))
for k in all_keys:
    g = v_comp_good.raw_scores.get(k, 0)
    b = v_comp_bad.raw_scores.get(k, 0)
    print(f"  {k:<18s} {g:>12.4f} {b:>12.4f}")

Metric                 Good Trace    Bad Trace
----------------------------------------------
passed                       True        False
confidence                  1.000        0.040
  constraint.checks_passed       5.0000       1.0000
  constraint.total_checks       5.0000       5.0000
  constraint.violations       0.0000       4.0000
  exact_outcome.exact_match       1.0000       0.0000
  trace.checks             2.0000       2.0000
  trace.violations         0.0000       2.0000


---

## The make_oracle() Factory

The `make_oracle()` factory creates oracles from dictionary definitions, enabling YAML-driven configuration. Composite oracles are resolved recursively.

In [12]:
# Create a composite oracle from a dictionary definition
oracle_def = {
    "type": "composite",
    "threshold": 0.5,
    "fail_on_any_critical": True,
    "oracles": [
        {"type": "exact_outcome", "weight": 0.5},
        {"type": "constraint", "weight": 0.2},
        {"type": "trace", "weight": 0.3},
    ],
}

oracle_from_def = make_oracle(oracle_def, store=store)
v_factory = oracle_from_def.evaluate(scenario, good_trace)

matches = (v_factory.passed == v_comp_good.passed and
           abs(v_factory.confidence - v_comp_good.confidence) < 0.001)

print("=== make_oracle() from dict ===")
print(f"  oracle_type: {v_factory.oracle_type}")
print(f"  passed     : {v_factory.passed}")
print(f"  confidence : {v_factory.confidence:.3f}")
print(f"  Matches direct construction: {matches}")

=== make_oracle() from dict ===
  oracle_type: composite
  passed     : True
  confidence : 1.000
  Matches direct construction: True


---

## Lab 15.2 — Oracles with Live Agent Traces

In this lab we run the `FullMockHarness` through the adapter, capture real traces and evaluate them with the composite oracle.

In [ ]:
from importlib.resources import files
# Lab 15.2a — Run agent and evaluate with oracles
from knowlytix.harness.testing.adapters import AgentAdapter
from knowlytix.harness.fixtures import FullMockHarness, create_mock_environment

env = create_mock_environment()
adapter = AgentAdapter(
    harness=FullMockHarness(environment=env, recovery_mode="fallback"),
)

context = open(str(files("knowlytix.harness.testing").joinpath("tests/model_risk_assessment.md"))).read()

# Run a single question
env.reset(seed=42)
state_before = env.get_state()
trace_live = adapter.evaluate(
    question="What is the risk rating of MDL-001?",
    context=context,
)
state_after = env.get_state()

print(f"Answer: {trace_live.raw_output[:80]!r}")
print(f"Tool calls: {[tc.tool_name for tc in trace_live.tool_calls]}")
print(f"Latency: {trace_live.latency_ms:.2f} ms")

In [14]:
# Lab 15.2b — Evaluate the live trace with all oracle types
# Build scenario matching the question
live_scenario = Scenario(
    scenario_id="LIVE-001",
    name="MDL-001 risk rating (live)",
    question_text="What is the risk rating of MDL-001?",
    ground_truth="High",
    metadata={"acceptable_set": ["High", "high"]},
    trace_pattern="data_lookup, risk_check, !dangerous_delete",
    expected_tool_sequence=["data_lookup", "risk_check"],
    constraints=[
        {"type": "max_tool_calls", "value": 10},
        {"type": "forbidden_tools", "value": ["dangerous_delete"]},
    ],
    expected_side_effects={"tickets_created": "__unchanged__"},
)

oracles = [
    ("ExactOutcome", ExactOutcomeOracle(store=store)),
    ("AcceptableSet", AcceptableSetOracle(store=store)),
    ("Constraint", ConstraintOracle(store=store)),
    ("Trace", TraceOracle(store=store)),
    ("SideEffect", SideEffectOracle(environment=env, store=store)),
]

print(f"{'Oracle':<20s} {'Passed':<8s} {'Confidence':>10s} {'Detail'}")
print("-" * 80)
for name, oracle in oracles:
    v = oracle.evaluate(live_scenario, trace_live, state_before, state_after)
    print(f"{name:<20s} {str(v.passed):<8s} {v.confidence:>10.3f} {v.detail[:40]}")

Oracle               Passed   Confidence Detail
--------------------------------------------------------------------------------
ExactOutcome         False         0.000 exact_match=False, gms_score=None
AcceptableSet        False         0.000 best_match=None, score=0.000, checked 2 
Constraint           True          1.000 2/2 constraints satisfied
Trace                True          1.000 trace: 3 calls, 0 violations
SideEffect           True          1.000 1/1 side effects verified


In [15]:
# Lab 15.2c — Full composite oracle on live trace
live_composite = CompositeOracle(
    oracles=[
        (ExactOutcomeOracle(store=store), 0.4),
        (ConstraintOracle(store=store), 0.2),
        (TraceOracle(store=store), 0.2),
        (SideEffectOracle(environment=env, store=store), 0.2),
    ],
    threshold=0.5,
    fail_on_any_critical=True,
)

v_live = live_composite.evaluate(live_scenario, trace_live,
                                  state_before, state_after)
print("=== Composite Verdict (live trace) ===")
print(f"  passed     : {v_live.passed}")
print(f"  confidence : {v_live.confidence:.3f}")
print(f"  detail     : {v_live.detail}")

=== Composite Verdict (live trace) ===
  passed     : False
  confidence : 0.600
  detail     : composite(4 oracles): avg_conf=0.600, threshold=0.5 | exact_outcome: FAIL (conf=0.000); constraint: PASS (conf=1.000); trace: PASS (conf=1.000); side_effect: PASS (conf=1.000)


---

## End-to-End: 640 Questions with Oracle Evaluation

We run the full DOE benchmark and evaluate every trace with a composite oracle. This gives us pass/fail verdicts across the full factor space.

In [16]:
# E2E: 640 questions with oracle evaluation
e2e_env = create_mock_environment()
e2e_adapter = AgentAdapter(
    harness=FullMockHarness(environment=e2e_env, recovery_mode="fallback"),
)

e2e_composite = CompositeOracle(
    oracles=[
        (ExactOutcomeOracle(store=store), 0.4),
        (ConstraintOracle(store=store), 0.3),
        (TraceOracle(store=store), 0.3),
    ],
    threshold=0.5,
    fail_on_any_critical=False,  # don't fail on sub-oracle failures
)

# Build a generic scenario for DOE questions
doe_scenario = Scenario(
    scenario_id="DOE-generic",
    constraints=[
        {"type": "max_tool_calls", "value": 10},
        {"type": "forbidden_tools", "value": ["dangerous_delete"]},
    ],
    trace_pattern="data_lookup, !dangerous_delete",
)

doe_questions = bench.generate_questions()
verdicts = []
for dq in doe_questions:
    # Update scenario with question-specific ground truth
    doe_scenario.ground_truth = dq.question.ground_truth
    doe_scenario.question_text = dq.question.natural_language

    t = e2e_adapter.evaluate(
        question=dq.question.natural_language,
        context=context,
    )
    v = e2e_composite.evaluate(doe_scenario, t)
    verdicts.append(v)

print(f"Evaluated {len(verdicts)} traces")

  Base questions: 160 (20 generators, max 10/cat)

  DOE expansion: 160 base x 4 runs = 640 questions
  Factors: ['clarity', 'entity_aliasing', 'reasoning_cue']


Evaluated 640 traces


In [17]:
# Analyze oracle verdicts
n_pass = sum(1 for v in verdicts if v.passed)
n_fail = sum(1 for v in verdicts if not v.passed)
confs = [v.confidence for v in verdicts]

print(f"=== Oracle Verdict Summary ({len(verdicts)} questions) ===")
print(f"  Passed : {n_pass} ({100*n_pass/len(verdicts):.1f}%)")
print(f"  Failed : {n_fail} ({100*n_fail/len(verdicts):.1f}%)")
print("\n=== Confidence Distribution ===")
print(f"  Mean   : {np.mean(confs):.3f}")
print(f"  Std    : {np.std(confs):.3f}")
print(f"  Median : {np.median(confs):.3f}")
print(f"  Min    : {np.min(confs):.3f}")
print(f"  Max    : {np.max(confs):.3f}")

=== Oracle Verdict Summary (640 questions) ===
  Passed : 640 (100.0%)
  Failed : 0 (0.0%)

=== Confidence Distribution ===
  Mean   : 0.612
  Std    : 0.070
  Median : 0.600
  Min    : 0.600
  Max    : 1.000


In [18]:
# Per-oracle score distribution
oracle_scores = {}
for v in verdicts:
    for k, s in v.raw_scores.items():
        oracle_scores.setdefault(k, []).append(s)

print("=== Per-Oracle Scores ===")
print(f"{'Score Key':<35s} {'Mean':>8s} {'Std':>8s} {'Min':>8s} {'Max':>8s}")
print("-" * 70)
for k in sorted(oracle_scores.keys()):
    vals = oracle_scores[k]
    print(f"{k:<35s} {np.mean(vals):>8.4f} {np.std(vals):>8.4f} "
          f"{np.min(vals):>8.4f} {np.max(vals):>8.4f}")

=== Per-Oracle Scores ===
Score Key                               Mean      Std      Min      Max
----------------------------------------------------------------------
constraint.checks_passed              2.0000   0.0000   2.0000   2.0000
constraint.total_checks               2.0000   0.0000   2.0000   2.0000
constraint.violations                 0.0000   0.0000   0.0000   0.0000
exact_outcome.exact_match             0.0312   0.1740   0.0000   1.0000
trace.checks                          2.0000   0.0000   2.0000   2.0000
trace.violations                      0.0000   0.0000   0.0000   0.0000


---

## Diagnostic Pitfall: Lenient Composites Mask Correctness Failures

The previous E2E run reports 100% pass — yet `exact_outcome.exact_match` has mean ≈ 0.008. How can both be true?

Two knobs did the masking:

1. **`fail_on_any_critical=False`** — a sub-oracle failure never short-circuits the composite. The verdict is a weighted average only.
2. **Weak constraint / trace sub-oracles** — the generic DOE scenario declares only two cheap constraints and a permissive `trace_pattern`, so both oracles auto-pass near 1.0 on almost every trace. At weights 0.3 + 0.3 = 0.6 they already clear the 0.5 threshold on their own, regardless of correctness.

The fix is not to calibrate harder — it is to choose a composite whose weights and critical-failure policy match the question you are asking the oracle. Below we rerun the same 520 traces under two stricter composites and compare.


In [ ]:
# Rerun the same 520 traces under stricter composite policies
#   A. lenient    — weights (0.4, 0.3, 0.3), fail_on_any_critical=False  (the original)
#   B. strict     — same weights, fail_on_any_critical=True
#   C. correctness-weighted — weights (0.7, 0.15, 0.15), fail_on_any_critical=False
#
# We evaluate each trace with all three composites to get directly comparable
# pass rates without re-running the agent.

lenient = CompositeOracle(
    oracles=[
        (ExactOutcomeOracle(store=store), 0.4),
        (ConstraintOracle(store=store), 0.3),
        (TraceOracle(store=store), 0.3),
    ],
    threshold=0.5,
    fail_on_any_critical=False,
)
strict = CompositeOracle(
    oracles=[
        (ExactOutcomeOracle(store=store), 0.4),
        (ConstraintOracle(store=store), 0.3),
        (TraceOracle(store=store), 0.3),
    ],
    threshold=0.5,
    fail_on_any_critical=True,
)
correctness_weighted = CompositeOracle(
    oracles=[
        (ExactOutcomeOracle(store=store), 0.7),
        (ConstraintOracle(store=store), 0.15),
        (TraceOracle(store=store), 0.15),
    ],
    threshold=0.5,
    fail_on_any_critical=False,
)

# Walk the DOE questions once more, scoring every trace against all three.
lenient_v, strict_v, cw_v = [], [], []
for dq in doe_questions:
    doe_scenario.ground_truth = dq.question.ground_truth
    doe_scenario.question_text = dq.question.natural_language
    t = e2e_adapter.evaluate(
        question=dq.question.natural_language,
        context=context,
    )
    lenient_v.append(lenient.evaluate(doe_scenario, t))
    strict_v.append(strict.evaluate(doe_scenario, t))
    cw_v.append(correctness_weighted.evaluate(doe_scenario, t))

n = len(lenient_v)
rows = [
    ("lenient (0.4/0.3/0.3, any_critical=F)", lenient_v),
    ("strict  (0.4/0.3/0.3, any_critical=T)", strict_v),
    ("correctness-weighted (0.7/0.15/0.15)", cw_v),
]
print(f"{'Composite policy':<42s} {'Pass %':>8s} {'Mean conf':>10s}")
print("-" * 62)
for label, vs in rows:
    pr = 100 * sum(1 for v in vs if v.passed) / n
    mc = sum(v.confidence for v in vs) / n
    print(f"{label:<42s} {pr:>7.1f}% {mc:>10.3f}")

# Correctness baseline for reference (exact-match-only decision).
exact_only = sum(
    1 for v in verdicts
    if v.raw_scores.get('exact_outcome.exact_match', 0.0) >= 0.5
)
print(f"\nFor reference: exact-match-only pass rate = "
      f"{100*exact_only/len(verdicts):.1f}% ({exact_only}/{len(verdicts)})")
print("Lenient composite hides this almost entirely; strict mode exposes it.")

---

## YAML-Configurable Weights — Aggregation as a Config Artifact

Chapter 9 gave us a YAML knob for `TypedVerdict.overall_confidence` weights; Chapter 10 did the same for severity composite weights. Composite-oracle weights complete the triad — the same `severity_config.yaml` can drive all three aggregation layers.

Two concrete benefits:

1. **Defaults are explicit, overrides are localized.** `DEFAULT_COMPOSITE_ORACLE_WEIGHTS` lives in `knowlytix/harness/testing/oracles.py`; the YAML file only names the keys you override. Unknown keys raise `ValueError`, so a typo can't silently fall back to defaults.
2. **Per-sub-oracle transparency.** `CompositeOracle.evaluate` now exposes `{type}.passed`, `{type}.confidence` and `{type}.weight` in `raw_scores` for every sub-oracle. The weighted aggregate is always auditable against its inputs.


In [ ]:
# Load YAML-driven weights and build a CompositeOracle via from_weights()
from knowlytix.harness.testing import (
    DEFAULT_COMPOSITE_ORACLE_WEIGHTS,
    load_composite_oracle_weights,
)

yaml_weights = load_composite_oracle_weights(
    "knowlytix/harness/testing/severity_config.example.yaml"
)
print("Defaults vs YAML override:")
print(f"{'oracle_type':<18s} {'default':>8s} {'yaml':>8s}")
print("-" * 36)
for k in sorted(DEFAULT_COMPOSITE_ORACLE_WEIGHTS):
    d = DEFAULT_COMPOSITE_ORACLE_WEIGHTS[k]
    y = yaml_weights[k]
    mark = "  *" if abs(d - y) > 1e-9 else ""
    print(f"{k:<18s} {d:>8.2f} {y:>8.2f}{mark}")

# Build the composite from the YAML weights.  Zero-weight oracles
# (acceptable_set, human_reviewed) are skipped automatically.
yaml_composite = CompositeOracle.from_weights(
    yaml_weights,
    store=store,
    threshold=0.5,
    fail_on_any_critical=False,
)
print(f"\nActive sub-oracles: "
      f"{[o.oracle_type for o, _ in yaml_composite._oracles]}")

# Re-evaluate the same good / bad traces and show the transparent
# per-sub-oracle raw_scores.
v_yaml = yaml_composite.evaluate(scenario, good_trace)
print("\n=== YAML composite on good_trace ===")
print(f"  passed     : {v_yaml.passed}")
print(f"  confidence : {v_yaml.confidence:.3f}")
print("  per-sub-oracle audit trail:")
audit_keys = sorted(
    k for k in v_yaml.raw_scores
    if k.endswith(".passed") or k.endswith(".confidence")
    or k.endswith(".weight")
)
for k in audit_keys:
    print(f"    {k:<32s} {v_yaml.raw_scores[k]:>8.3f}")

---

## Exercises

**Exercise 15.1** — Create an `AcceptableSetOracle` with five acceptable answers (e.g. `["High", "Elevated", "Critical", "Above threshold", "Significant"]`). Evaluate three traces with different outputs and verify that partial matches (containment) work as expected.

**Exercise 15.2** — Build a `ConstraintOracle` scenario with six constraints including `max_latency_ms=2.0` and `output_max_length=500`. Run the 640-question benchmark and report the violation rate for each constraint type.

**Exercise 15.3** — Construct a `CompositeOracle` with all five oracle types (exact, constraint, trace, side-effect and acceptable-set) with custom weights summing to 1.0. Compare `fail_on_any_critical=True` vs `False` on the 640-question benchmark. How many verdicts flip?

---

## Summary

This chapter introduced the **Oracle Framework** — the final layer of the black-box testing stack that judges whether agent behavior is acceptable:

1. **OracleVerdict** provides a uniform result schema: pass/fail, confidence, human-readable detail, evidence items and raw scores.

2. **Seven oracle types** each leverage a different GMS primitive:
   - `ExactOutcomeOracle` uses `score_triple()` for geodesic answer matching
   - `AcceptableSetOracle` finds the nearest acceptable answer by geodesic distance
   - `ConstraintOracle` evaluates must-have/must-not-have predicates on output, tools and state
   - `TraceOracle` verifies tool-call sequences with ordering constraints and `check_holonomy()`
   - `SideEffectOracle` compares environment state changes against expectations using `tension_energy()`
   - `HumanReviewedOracle` drives rubric-based LLM/human evaluation
   - `CompositeOracle` computes weighted verdicts from multiple sub-oracles

3. **Composition** via `CompositeOracle` enables multi-signal evaluation. The `fail_on_any_critical` flag allows strict (any failure = composite failure) or lenient (weighted average only) modes.

4. **make_oracle()** factory creates oracles from dictionary definitions for YAML-driven configuration.

This chapter completes **Part IV: Black-Box Agentic Testing**. The full stack — adapter, gateway, policy, environment, oracles — provides end-to-end infrastructure for testing agents without access to their internals.

In Part V we turn to **Behavioral and Quality Testing**: hallucination detection, calibration, stateful multi-turn and multi-agent scenarios.